# 📷 Notebook Complementar — OpenCV para PLN
### Aula 2 — Tecnologias e Ferramentas

---

## 🎯 O que você vai aprender

- Como o OpenCV representa imagens como arrays NumPy
- Operações fundamentais de pré-processamento de imagens
- Detecção de texto em imagens (base para OCR)
- Como preparar imagens para modelos de deep learning
- Pipeline completo: imagem → pré-processamento → análise

---

## 📚 Teoria: Por que OpenCV no PLN?

O PLN moderno vai além do texto puro. Sistemas reais precisam processar:

- 📄 **Documentos escaneados** → OCR, extração de texto
- 📸 **Imagens com legendas** → Image Captioning
- 📹 **Vídeos com transcrição** → Speech-to-text + análise visual
- 🛒 **Produtos em fotos** → Identificação + descrição automática

O **OpenCV** é a ponte entre o mundo visual e o textual.

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# IMPORTAÇÕES
# ─────────────────────────────────────────────────────────────
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw, ImageFont
import warnings
warnings.filterwarnings('ignore')

# Função utilitária para exibir imagens no notebook
def mostrar(imagens, titulos, figsize=(14, 4), cmap_list=None):
    """Exibe uma lista de imagens lado a lado com títulos."""
    n = len(imagens)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for ax, img, titulo in zip(axes, imagens, titulos):
        cmap = 'gray' if len(img.shape) == 2 else None
        ax.imshow(img, cmap=cmap)
        ax.set_title(titulo, fontsize=10, fontweight='bold')
        h, w = img.shape[:2]
        ax.set_xlabel(f'{w}×{h} px', fontsize=9)
        ax.tick_params(left=False, bottom=False,
                        labelleft=False, labelbottom=False)
    plt.tight_layout()
    plt.show()

print(f'✅ OpenCV: {cv2.__version__}')
print(f'✅ NumPy : {np.__version__}')
print(f'✅ Pillow: {Image.__version__}')
print()
print('💡 Lembre: OpenCV usa BGR. Converter para RGB antes de exibir!')

---
## 🔵 Parte 1 — Representação de Imagens

### 1.1 Como o OpenCV armazena imagens

In [ ]:
# ─────────────────────────────────────────────────────────────
# REPRESENTAÇÃO DE IMAGENS
# Uma imagem é um array NumPy de shape (altura, largura, canais)
# Cada pixel tem 3 valores (B, G, R) entre 0 e 255
# ─────────────────────────────────────────────────────────────

# Criar uma imagem simples de 4×4 pixels manualmente
# Cada linha é uma linha de pixels, cada coluna um pixel [B, G, R]
img_pequena = np.array([
    [[255,   0,   0], [  0, 255,   0], [  0,   0, 255], [255, 255,   0]],  # azul, verde, vermelho, amarelo
    [[255,   0, 255], [  0, 255, 255], [255, 255, 255], [  0,   0,   0]],  # magenta, ciano, branco, preto
    [[128,   0,   0], [  0, 128,   0], [  0,   0, 128], [128, 128,   0]],  # tons mais escuros
    [[200, 100,  50], [ 50, 100, 200], [100, 200,  50], [150, 150, 150]],  # mistos
], dtype=np.uint8)  # dtype=uint8 → valores inteiros de 0 a 255

print('📐 Propriedades da imagem:')
print(f'   Shape  : {img_pequena.shape}  → (altura=4, largura=4, canais=3)')
print(f'   Dtype  : {img_pequena.dtype}  → valores inteiros 0-255')
print(f'   Memória: {img_pequena.nbytes} bytes = {img_pequena.nbytes/1024:.3f} KB')
print()
print('🎨 Valores do pixel [0,0] (topo-esquerda):')
print(f'   BGR = {img_pequena[0,0]}  → B={img_pequena[0,0,0]}, G={img_pequena[0,0,1]}, R={img_pequena[0,0,2]}')
print()
print('⚠️  OpenCV armazena em BGR (Blue, Green, Red)')
print('   matplotlib e PIL usam RGB (Red, Green, Blue)')

# Visualizar a imagem pequena (ampliada para ver os pixels)
img_rgb = cv2.cvtColor(img_pequena, cv2.COLOR_BGR2RGB)  # converter BGR → RGB
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# Original RGB
axes[0].imshow(img_rgb, interpolation='nearest')
axes[0].set_title('Imagem RGB (correta)', fontweight='bold')
axes[0].axis('off')

# Se exibir sem converter (BGR → cores trocadas)
axes[1].imshow(img_pequena, interpolation='nearest')
axes[1].set_title('Sem converter BGR (cores erradas!)', fontweight='bold', color='red')
axes[1].axis('off')

# Canais separados
canal_b = img_pequena[:,:,0]  # Canal Blue (índice 0 no BGR)
axes[2].imshow(canal_b, cmap='Blues', interpolation='nearest')
axes[2].set_title('Canal Blue isolado', fontweight='bold')
axes[2].axis('off')

plt.suptitle('Representação de Imagens no OpenCV', fontweight='bold')
plt.tight_layout()
plt.show()

---
### 1.2 Criando Imagens Sintéticas para PLN

In [ ]:
# ─────────────────────────────────────────────────────────────
# CRIANDO IMAGENS COM TEXTO
# cv2.putText() → escreve texto em uma imagem
# Parâmetros: img, texto, posição, fonte, escala, cor, espessura
# ─────────────────────────────────────────────────────────────

def criar_documento_sintetico(largura=600, altura=400):
    """Cria um documento sintético com texto para demonstrações."""
    img = np.ones((altura, largura, 3), dtype=np.uint8) * 250  # fundo quase branco

    # Borda do documento
    cv2.rectangle(img, (10, 10), (largura-10, altura-10), (200, 200, 200), 1)

    # Título
    cv2.putText(img, 'RELATORIO TRIMESTRAL - PLN Corp.', (30, 50),
                cv2.FONT_HERSHEY_DUPLEX, 0.7, (30, 30, 30), 1)
    cv2.line(img, (30, 60), (largura-30, 60), (180, 180, 180), 1)

    # Conteúdo
    linhas = [
        'Data: 28/04/2025',
        'Empresa: PLN Tecnologia S.A.',
        'Setor: Inteligencia Artificial',
        '',
        'Resumo Executivo:',
        'O processamento de linguagem natural cresceu 42% no trimestre.',
        'Novos modelos foram implementados com sucesso nas operacoes.',
        'A acuracia dos sistemas atingiu 97.3% nas tarefas de NER.',
        '',
        'Contato: contato@plncorp.com.br | Tel: (11) 4002-8922',
    ]

    y = 90
    for linha in linhas:
        cor = (50, 100, 180) if ':' in linha and linha.endswith(':') else (40, 40, 40)
        negrito = cv2.FONT_HERSHEY_DUPLEX if linha.endswith(':') else cv2.FONT_HERSHEY_SIMPLEX
        cv2.putText(img, linha, (30, y), negrito, 0.48, cor, 1)
        y += 30

    # Elementos gráficos: tabela simples
    cv2.rectangle(img, (30, 330), (570, 380), (220, 220, 240), -1)
    cv2.rectangle(img, (30, 330), (570, 380), (180, 180, 200), 1)
    cv2.putText(img, 'Q1: R$1.2M  |  Q2: R$1.8M  |  Q3: R$2.1M  |  Total: R$5.1M',
                (45, 360), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (30, 30, 100), 1)

    return img

documento = criar_documento_sintetico()
doc_rgb   = cv2.cvtColor(documento, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(9, 6))
plt.imshow(doc_rgb)
plt.title('Documento Sintético — Base para Demonstrações de PLN', fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f'📐 Shape do documento: {documento.shape}')

---
## 🟢 Parte 2 — Pipeline de Pré-processamento

### 2.1 Etapas de preparação para modelos de PLN

In [ ]:
# ─────────────────────────────────────────────────────────────
# PIPELINE COMPLETO DE PRÉ-PROCESSAMENTO
# Transforma um documento real em dados prontos para ML
# ─────────────────────────────────────────────────────────────

# Adicionar ruído gaussiano ao documento (simula scanner de baixa qualidade)
ruido = np.random.normal(0, 22, documento.shape).astype(np.int16)
doc_ruidoso = np.clip(documento.astype(np.int16) + ruido, 0, 255).astype(np.uint8)

# ── ETAPA 1: Converter para escala de cinza
# Reduz 3 canais (BGR) para 1 canal — mais simples para OCR
doc_cinza = cv2.cvtColor(doc_ruidoso, cv2.COLOR_BGR2GRAY)

# ── ETAPA 2: Suavização Gaussiana (reduz ruído)
# Kernel (5,5) → área de influência. Maior kernel = mais suavização
doc_suave = cv2.GaussianBlur(doc_cinza, (5, 5), 0)

# ── ETAPA 3: Binarização (OTSU)
# Converte pixels em preto (0) ou branco (255)
# OTSU encontra automaticamente o threshold ideal
_, doc_binario = cv2.threshold(
    doc_suave,
    0,           # threshold ignorado com OTSU
    255,         # valor máximo
    cv2.THRESH_BINARY + cv2.THRESH_OTSU  # método automático
)

# ── ETAPA 4: Dilatação (reforça o texto)
# Morfologia: expande as regiões brancas
kernel = np.ones((2, 2), np.uint8)
doc_dilatado = cv2.dilate(doc_binario, kernel, iterations=1)

# ── ETAPA 5: Redimensionar para tamanho padrão
doc_resize = cv2.resize(doc_dilatado, (448, 300))

# ── ETAPA 6: Normalizar [0,1] para redes neurais
doc_norm = doc_resize.astype(np.float32) / 255.0

# Visualizar todas as etapas
etapas = [
    (cv2.cvtColor(doc_ruidoso, cv2.COLOR_BGR2RGB), '1. Com Ruído (simulando scanner)'),
    (doc_cinza,    '2. Escala de Cinza'),
    (doc_suave,    '3. Suavizado (Gaussiano 5×5)'),
    (doc_binario,  '4. Binarizado (Threshold OTSU)'),
    (doc_dilatado, '5. Dilatado (texto reforçado)'),
    (doc_resize,   '6. Redimensionado (448×300)'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (img, titulo) in zip(axes.flatten(), etapas):
    cmap = 'gray' if len(img.shape) == 2 else None
    ax.imshow(img, cmap=cmap)
    ax.set_title(titulo, fontsize=10, fontweight='bold')
    h, w = img.shape[:2]
    ax.set_xlabel(f'{w}×{h} px', fontsize=9)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

plt.suptitle('Pipeline de Pré-processamento de Documento para PLN',
             fontweight='bold', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('📊 Resumo do Pipeline:')
print(f'   Entrada  : {doc_ruidoso.shape}  (BGR, uint8, 0-255)')
print(f'   Saída    : {doc_norm.shape}   (cinza, float32, 0.0-1.0)')
print(f'   Memória  : {doc_ruidoso.nbytes:,} bytes → {doc_norm.nbytes:,} bytes')

---
### 2.2 Transformações Morfológicas — Isolando Regiões de Texto

In [ ]:
# ─────────────────────────────────────────────────────────────
# MORFOLOGIA MATEMÁTICA
# Operações que modificam a forma de regiões na imagem
# Usadas para limpar e realçar texto em documentos
# ─────────────────────────────────────────────────────────────

# Trabalhar com o documento binarizado
base = doc_binario.copy()

# Definir kernels de diferentes tamanhos
k3 = np.ones((3, 3), np.uint8)   # pequeno
k7 = np.ones((7, 7), np.uint8)   # médio

# ── Erosão: encolhe regiões brancas (remove ruído fino)
erosao = cv2.erode(base, k3, iterations=1)

# ── Dilatação: expande regiões brancas (une fragmentos)
dilatacao = cv2.dilate(base, k3, iterations=1)

# ── Abertura: erosão + dilatação (remove ruído pequeno)
abertura = cv2.morphologyEx(base, cv2.MORPH_OPEN, k3)

# ── Fechamento: dilatação + erosão (preenche buracos no texto)
fechamento = cv2.morphologyEx(base, cv2.MORPH_CLOSE, k3)

# ── Gradiente morfológico: diferença entre dilatação e erosão
# Resulta nas bordas/contornos das regiões
gradiente = cv2.morphologyEx(base, cv2.MORPH_GRADIENT, k3)

# ── Kernel horizontal (detecta linhas de texto)
k_horizontal = np.ones((1, 30), np.uint8)
linhas_texto = cv2.morphologyEx(base, cv2.MORPH_CLOSE, k_horizontal)

ops = [
    (base,          'Original (binarizado)'),
    (erosao,        'Erosão (encolhe)'),
    (dilatacao,     'Dilatação (expande)'),
    (abertura,      'Abertura (remove ruído)'),
    (fechamento,    'Fechamento (une texto)'),
    (linhas_texto,  'Linhas de texto detectadas'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (img, titulo) in zip(axes.flatten(), ops):
    ax.imshow(img, cmap='gray')
    ax.set_title(titulo, fontsize=10, fontweight='bold')
    ax.axis('off')

plt.suptitle('Operações Morfológicas para Pré-processamento de Texto',
             fontweight='bold', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('💡 Uso prático:')
print('   Abertura  → limpar ruído antes de OCR')
print('   Fechamento → unir fragmentos de letras cortadas')
print('   Linhas    → detectar blocos de texto em documentos')

---
## 🔴 Parte 3 — Detecção de Regiões de Texto

### 3.1 Encontrando Contornos de Texto

In [ ]:
# ─────────────────────────────────────────────────────────────
# DETECÇÃO DE REGIÕES DE TEXTO COM CONTORNOS
# Encontra 'caixas' ao redor de cada bloco de texto
# Base para sistemas de OCR e análise de layout
#
# cv2.findContours() → encontra contornos na imagem binarizada
# cv2.boundingRect() → retorna (x, y, largura, altura) do contorno
# ─────────────────────────────────────────────────────────────

# Preparar imagem: inverter (texto preto em fundo branco → branco em preto)
doc_trabalho = doc_binario.copy()
doc_invertido = cv2.bitwise_not(doc_trabalho)

# Dilatar para unir letras próximas em blocos de palavras
k_palavra = np.ones((3, 12), np.uint8)
doc_palavras = cv2.dilate(doc_invertido, k_palavra, iterations=2)

# Encontrar contornos
contornos, hierarquia = cv2.findContours(
    doc_palavras,
    cv2.RETR_EXTERNAL,    # apenas contornos externos
    cv2.CHAIN_APPROX_SIMPLE  # compressão de pontos redundantes
)

print(f'🔍 Contornos encontrados: {len(contornos)}')

# Desenhar bounding boxes na imagem original
doc_deteccao = cv2.cvtColor(doc_trabalho, cv2.COLOR_GRAY2BGR)
regioes_validas = []

for i, contorno in enumerate(contornos):
    x, y, w, h = cv2.boundingRect(contorno)

    # Filtrar regiões muito pequenas (ruído) e muito grandes (fundo)
    area = w * h
    if 200 < area < 60000 and w > 20 and h > 8:
        regioes_validas.append((x, y, w, h))

        # Colorir por tamanho de região
        if area > 10000:
            cor = (255, 50, 50)   # vermelho = região grande (linha de texto)
        elif area > 2000:
            cor = (50, 180, 50)   # verde = palavra
        else:
            cor = (50, 100, 255)  # azul = caractere/fragmento

        cv2.rectangle(doc_deteccao, (x, y), (x+w, y+h), cor, 1)

print(f'✅ Regiões válidas (após filtro de área e dimensão): {len(regioes_validas)}')

# Visualizar
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(cv2.cvtColor(documento, cv2.COLOR_BGR2RGB))
axes[0].set_title('1. Documento Original', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(doc_palavras, cmap='gray')
axes[1].set_title('2. Pós-dilatação (agrupa palavras)', fontweight='bold')
axes[1].axis('off')

axes[2].imshow(cv2.cvtColor(doc_deteccao, cv2.COLOR_BGR2RGB))
axes[2].set_title(f'3. {len(regioes_validas)} Regiões Detectadas\n(azul=char, verde=palavra, vermelho=linha)',
                   fontweight='bold')
axes[2].axis('off')

plt.suptitle('Detecção de Regiões de Texto em Documentos', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
### 3.2 Recortando e Analisando Regiões

Após detectar as regiões, podemos recortá-las para envio a um modelo de OCR.

In [ ]:
# ─────────────────────────────────────────────────────────────
# RECORTE DE REGIÕES DETECTADAS
# Cada região detectada pode ser enviada a um modelo de OCR
#
# img[y:y+h, x:x+w] → recorte (slice) da imagem NumPy
# ─────────────────────────────────────────────────────────────

# Ordenar regiões de cima para baixo (ordem de leitura)
regioes_ordenadas = sorted(regioes_validas, key=lambda r: r[1])  # sort by y

# Selecionar as 6 maiores regiões (mais prováveis de serem linhas de texto)
regioes_grandes = sorted(regioes_ordenadas, key=lambda r: r[2]*r[3], reverse=True)[:6]
regioes_grandes = sorted(regioes_grandes, key=lambda r: r[1])  # re-ordenar por y

print(f'📋 Mostrando as {len(regioes_grandes)} maiores regiões detectadas:\n')

if len(regioes_grandes) > 0:
    n_cols = min(3, len(regioes_grandes))
    n_rows = (len(regioes_grandes) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 2.5))

    if n_rows * n_cols == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes.reshape(1, -1)

    for idx, (x, y, w, h) in enumerate(regioes_grandes):
        # Recortar a região do documento original
        recorte = documento[y:y+h, x:x+w]
        recorte_rgb = cv2.cvtColor(recorte, cv2.COLOR_BGR2RGB)

        row, col = divmod(idx, n_cols)
        ax = axes[row, col]
        ax.imshow(recorte_rgb)
        ax.set_title(f'Região {idx+1} | {w}×{h}px | área={w*h:,}',
                     fontsize=9, fontweight='bold')
        ax.axis('off')

        print(f'  Região {idx+1}: x={x}, y={y}, w={w}, h={h}  →  pronto para OCR')

    # Desativar eixos extras
    for idx in range(len(regioes_grandes), n_rows * n_cols):
        row, col = divmod(idx, n_cols)
        axes[row, col].axis('off')

    plt.suptitle('Regiões Recortadas — Prontas para Envio ao OCR',
                 fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print('Nenhuma região válida encontrada.')

---
## 🟡 Parte 4 — Preparando Imagens para Modelos de Deep Learning

### 4.1 Normalização e Padronização

In [ ]:
# ─────────────────────────────────────────────────────────────
# PREPARAÇÃO PARA DEEP LEARNING
# Modelos como VGG, ResNet e ViT esperam imagens em formatos
# específicos. Esta célula demonstra as transformações necessárias.
# ─────────────────────────────────────────────────────────────

# Parâmetros padrão de modelos famosos
configs_modelos = {
    'ResNet-50':    {'size': (224, 224), 'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225]},
    'ViT-B/16':     {'size': (224, 224), 'mean': [0.5,   0.5,   0.5  ], 'std': [0.5,   0.5,   0.5  ]},
    'EfficientNet': {'size': (300, 300), 'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225]},
    'CLIP':         {'size': (224, 224), 'mean': [0.481, 0.458, 0.408], 'std': [0.269, 0.261, 0.276]},
}

def preparar_para_modelo(img_bgr, config):
    """
    Prepara uma imagem BGR para uso em um modelo de deep learning.
    Aplica: resize → RGB → float32 → normalização por canal.
    """
    # 1. Redimensionar
    img_resize = cv2.resize(img_bgr, config['size'])

    # 2. BGR → RGB
    img_rgb = cv2.cvtColor(img_resize, cv2.COLOR_BGR2RGB)

    # 3. Converter para float32 e normalizar [0, 1]
    img_float = img_rgb.astype(np.float32) / 255.0

    # 4. Normalização por canal (subtrai média, divide por desvio)
    mean = np.array(config['mean'], dtype=np.float32)
    std  = np.array(config['std'],  dtype=np.float32)
    img_norm = (img_float - mean) / std

    return img_resize, img_rgb, img_float, img_norm

print('📊 Preparação de imagem para diferentes modelos:\n')
print(f'{"Modelo":<15} {"Tamanho":<12} {"Média canais":<30} {"Desvio padrão"}')
print('-' * 75)
for nome, config in configs_modelos.items():
    print(f'{nome:<15} {str(config["size"]):<12} {str(config["mean"]):<30} {config["std"]}')

# Demonstrar com ResNet
config_resnet = configs_modelos['ResNet-50']
resize, rgb_img, float_img, norm_img = preparar_para_modelo(documento, config_resnet)

print(f'\n🔬 Estatísticas após normalização (ResNet-50):')
for c, nome_canal in enumerate(['R', 'G', 'B']):
    canal = norm_img[:,:,c]
    print(f'   Canal {nome_canal}: min={canal.min():.3f} | max={canal.max():.3f} | '
          f'média={canal.mean():.3f} | std={canal.std():.3f}')

# Visualizar efeito da normalização
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(rgb_img)
axes[0].set_title('RGB Original\n[0, 255] uint8', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(np.clip(float_img, 0, 1))
axes[1].set_title('Float32\n[0.0, 1.0]', fontweight='bold')
axes[1].axis('off')

# Normalizada: clipar para visualização
viz_norm = (norm_img - norm_img.min()) / (norm_img.max() - norm_img.min())
axes[2].imshow(np.clip(viz_norm, 0, 1))
axes[2].set_title('Normalizado (ResNet)\n[-2.1, +2.6] float32', fontweight='bold')
axes[2].axis('off')

plt.suptitle('Transformações de Imagem para Deep Learning', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🟣 Parte 5 — Pipeline Completo: Imagem → Análise PLN

In [ ]:
# ─────────────────────────────────────────────────────────────
# PIPELINE COMPLETO: IMAGEM → PLN
# Simula um sistema completo de análise de documentos:
#   1. Recebe a imagem
#   2. Pré-processa
#   3. Detecta regiões de texto
#   4. Extrai metadados
#   5. Gera relatório estruturado
# ─────────────────────────────────────────────────────────────

def pipeline_analise_documento(img_bgr):
    """
    Pipeline completo de análise de documento.
    Retorna um dicionário com metadados estruturados.
    """
    relatorio = {}

    # ── ETAPA 1: Informações básicas
    h, w, c = img_bgr.shape
    relatorio['dimensoes'] = {'altura': h, 'largura': w, 'canais': c}
    relatorio['tamanho_kb'] = round(img_bgr.nbytes / 1024, 2)

    # ── ETAPA 2: Análise de cores
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    relatorio['brilho_medio'] = round(float(img_rgb.mean()), 2)
    relatorio['contraste']    = round(float(img_rgb.std()), 2)

    # ── ETAPA 3: Pré-processamento
    cinza     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    suave     = cv2.GaussianBlur(cinza, (3, 3), 0)
    _, binario = cv2.threshold(suave, 0, 255,
                               cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # ── ETAPA 4: Detecção de regiões
    invertido = cv2.bitwise_not(binario)
    k_linha   = np.ones((3, 15), np.uint8)
    dilatado  = cv2.dilate(invertido, k_linha, iterations=1)

    contornos, _ = cv2.findContours(
        dilatado, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    regioes = []
    for cont in contornos:
        x, y, rw, rh = cv2.boundingRect(cont)
        area = rw * rh
        if 500 < area < 80000 and rw > 40:
            regioes.append({'x':x, 'y':y, 'w':rw, 'h':rh, 'area':area})

    # Ordenar por posição vertical (ordem de leitura)
    regioes = sorted(regioes, key=lambda r: r['y'])
    relatorio['n_regioes_texto'] = len(regioes)
    relatorio['regioes']         = regioes

    # ── ETAPA 5: Estimativa de densidade de texto
    pixels_texto = np.sum(invertido > 0)
    pixels_total = h * w
    relatorio['densidade_texto'] = round(pixels_texto / pixels_total * 100, 2)

    return relatorio, binario

# Executar o pipeline
relatorio, binario = pipeline_analise_documento(documento)

# Exibir relatório
print('=' * 55)
print('  📋 RELATÓRIO DE ANÁLISE DE DOCUMENTO')
print('=' * 55)
print(f'  Dimensões      : {relatorio["dimensoes"]["largura"]}×{relatorio["dimensoes"]["altura"]} px')
print(f'  Tamanho        : {relatorio["tamanho_kb"]} KB')
print(f'  Brilho médio   : {relatorio["brilho_medio"]} / 255')
print(f'  Contraste      : {relatorio["contraste"]}')
print(f'  Regiões texto  : {relatorio["n_regioes_texto"]} blocos detectados')
print(f'  Densidade texto: {relatorio["densidade_texto"]}% da área total')
print()
print('  Regiões por ordem de leitura (y crescente):')
for i, reg in enumerate(relatorio['regioes'][:5]):
    print(f'   [{i+1}] pos=({reg["x"]},{reg["y"]}) tam={reg["w"]}×{reg["h"]} área={reg["area"]:,}')
if len(relatorio['regioes']) > 5:
    print(f'   ... e mais {len(relatorio["regioes"])-5} regiões')
print('=' * 55)
print('\n✅ Documento analisado. Regiões prontas para OCR/PLN!')

---
## 📝 Resumo

| Operação OpenCV | Função | Uso no PLN |
|---|---|---|
| `cvtColor(BGR2GRAY)` | Converte para escala de cinza | Reduz dimensionalidade para OCR |
| `GaussianBlur` | Suaviza a imagem | Remove ruído de scanner |
| `threshold OTSU` | Binariza automaticamente | Separa texto do fundo |
| `dilate / erode` | Operações morfológicas | Une fragmentos, remove ruído |
| `findContours` | Detecta bordas de regiões | Localiza blocos de texto |
| `boundingRect` | Retângulo ao redor de contorno | Recorta regiões para OCR |
| `resize` | Redimensiona imagem | Padroniza entrada para modelos |
| `normalize / / 255` | Normaliza pixels [0,1] | Entrada para redes neurais |

---

## 🔜 Próximo Passo

Com as imagens pré-processadas, o próximo passo é enviá-las para modelos multimodais  
(como o CLIP do Hugging Face) que combinam visão e linguagem — tema da **Aula 5**.